In [ ]:
import matplotlib.pyplot as plt
import subprocess
import os
import json

goodpt = 0.75   # controls the bimodal distribution
values = []     # stores the % b>r for each proportion
N = 10  # number of runs per value of goodagproportion
path = "gameDumps"

# Step 1: run all the experiments
for i in range(11):
    goodagproportion = i / 10.0 # to get 0.0, 0.1, 0.2 etc

    for j in range(N):
        command = f"go run main.go -goodpt={goodpt} -goodagproportion={goodagproportion}"
        subprocess.run(command, shell=True)


# Step 2: gather the data
for folder in os.listdir(path):
    sumForAverage = 0
    for file in os.listdir(f"{path}/{folder}"):
        with open(f"{path}/{folder}/{file}", 'r') as f:
            data = json.load(f)

        total = 0
        true_count = 0

        for iteration in data["iteration"]:
            joining_decisions = iteration.get("joiningDecisions", {})
            for agent_id, flag in joining_decisions.items():
                total += 1
                if flag:
                    true_count += 1

        proportion = true_count / total
        sumForAverage += proportion

    average = sumForAverage / N 
    values.append(average)

# percent of agents with "good" platonic tendency
labels = ['0', '10', '20', '30', '40', '50', '60', '70', '80', '90', '100']

# Step 3: Visualise the data
plt.bar(labels, values)
plt.xlabel(f"% of agents with good platonic tendency ({goodpt})")
plt.ylabel(f"% of decisions where average bike trust > regime trust")
plt.title(f"% B>R with respect to agent distribution")
plt.show()


# Step 4: Cleanup
path = "gameDumps"
for folder in os.listdir(path):
    for file in os.listdir(f"{path}/{folder}"):
        os.remove(f"{path}/{folder}/{file}")
